# 02 · Data pipeline

Replaces `scripts/ingest.py` (plan Section 4):

1. **Scrape** Facebook Marketplace with the Apify actor (`startUrls`, `resultsLimit`, a spend
   cap). Raw results go to `data/raw/` (gitignored: seller names and URLs).
2. **Anonymize**: drop seller name, seller id and URL; keep city. Ids are a hash of the
   Facebook id, so they stay the same on every scrape and the hand labels keep matching.
3. **Identify** the card with the LLM, 20 listings per call (free-tier friendly).
4. **Scope filter**: drop graded cards, lots, non-English printings, unidentified cards.
5. **Prices**: resolve set names to TCGdex ids, fetch TCGplayer market prices once, cache
   them in `data/prices.json`.

Output: `data/listings.json`, `data/prices.json`, and `data/listings_review.csv` for the
hand check. The same functions serve the Merchant Agent's live search.

Fixed from `ingest.py`: the actor input (`searchQuery`/`maxItems` → `startUrls`/`resultsLimit`),
apify-client 3.x returning a Run object, no spend cap, descriptions saved as `"{'text': ...}"`,
row-number ids, missing prices becoming $0, one LLM call per listing, and TCGdex lookups
with raw model output ("4/102", set names) instead of resolved set ids.

In [ ]:
#| default_exp pipeline

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Data pipeline (Section 4): Facebook Marketplace listings through Apify -> anonymize ->
LLM card identification -> scope filter -> TCGdex market prices. Also loads the listing pool
the Merchant Agent searches, and serves its optional live search."""

import csv
import hashlib
import json
import re
import time
from dataclasses import dataclass, field
from decimal import Decimal
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus

import httpx
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

from src.config import env, repo_path
from src.llm import LLM, offline_handler
from src.schemas import CardListing, Condition, Variant
from src.tools.risk import RedFlag, parse_red_flags

## 1. Scrape

In [ ]:
#| export
class SourceError(RuntimeError):
    pass

In [ ]:
#| export
def search_url(city: str, term: str) -> str:
    return f"https://www.facebook.com/marketplace/{city}/search/?query={quote_plus(term)}"

In [ ]:
#| export
class MarketplaceScraper:
    """Runs the Apify Facebook Marketplace actor. Pass `client` to use a fake in tests."""

    def __init__(self, config: dict[str, Any], client=None):
        self.cfg = config
        self._client = client

    @property
    def client(self):
        if self._client is None:
            token = env("APIFY_API_TOKEN", "APIFY_TOKEN")
            if not token:
                raise SourceError("APIFY_API_TOKEN is missing. Add it to .env (see .env.example).")
            from apify_client import ApifyClient

            self._client = ApifyClient(token)
        return self._client

    def search(
        self,
        terms: list[str],
        limit: int,
        max_charge_usd: float,
        details: bool = True,
        city: str | None = None,
        save_as: str = "snapshot",
    ) -> list[dict]:
        """Scrape the search terms and return raw items. Raw items are saved to data/raw/,
        which is gitignored because they contain seller names and listing URLs."""
        s = self.cfg["scrape"]
        run_input = {
            "startUrls": [{"url": search_url(city or s["city"], t)} for t in terms],
            "resultsLimit": limit,
            "includeListingDetails": details,
        }
        try:
            run = self.client.actor(s["actor"]).call(
                run_input=run_input, max_total_charge_usd=Decimal(str(max_charge_usd))
            )
        except SourceError:
            raise
        except Exception as e:
            raise SourceError(f"Apify run failed: {e}") from e
        if run is None:
            raise SourceError("Apify run did not start. Check the token and credit in the console.")
        # apify-client 3.x returns a Run object; older versions returned a dict.
        dataset_id = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
        items = list(self.client.dataset(dataset_id).iterate_items())

        out_dir = repo_path(self.cfg["data"]["raw_dir"])
        out_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime("%Y%m%d_%H%M%S")
        (out_dir / f"{save_as}_{stamp}.json").write_text(
            json.dumps(items, indent=1, ensure_ascii=False), encoding="utf-8"
        )
        return items

In [ ]:
#| export
def latest_raw(config: dict[str, Any], prefix: str = "snapshot") -> Path | None:
    """Newest raw file in data/raw/ (to redo extraction without scraping again)."""
    files = sorted(repo_path(config["data"]["raw_dir"]).glob(f"{prefix}_*.json"))
    return files[-1] if files else None

## 2. Anonymize

In [ ]:
#| export
def first(item: dict, *paths: str):
    """First non-empty value among dotted paths, e.g. 'location.reverse_geocode.city'."""
    for p in paths:
        cur: Any = item
        for part in p.split("."):
            cur = cur.get(part) if isinstance(cur, dict) else None
            if cur is None:
                break
        if cur not in (None, ""):
            return cur
    return None

In [ ]:
#| export
def to_price(value) -> float | None:
    if isinstance(value, dict):
        value = value.get("amount") or value.get("formatted_amount")
    if isinstance(value, int | float):
        return float(value)
    if isinstance(value, str):
        m = re.search(r"\d[\d,]*(?:\.\d+)?", value)
        return float(m.group(0).replace(",", "")) if m else None
    return None

In [ ]:
#| export
def anon_id(source_id: str) -> str:
    """Stable anonymous id: the same listing gets the same id on every scrape, so hand
    labels and the answer key stay valid, but the Facebook id itself is not stored."""
    return "fb_" + hashlib.sha256(str(source_id).encode()).hexdigest()[:10]

In [ ]:
#| export
def delivery_type(types: Any) -> str:
    text = " ".join(types).lower() if isinstance(types, list) else str(types or "").lower()
    if any(w in text for w in ("in_person", "pickup", "local")):
        return "in_person"  # shipping is $0 when pickup is offered
    if "ship" in text:
        return "shipping"
    return "unknown"

In [ ]:
#| export
def anonymize(item: dict) -> dict | None:
    """Map one raw Apify item to the fields we keep (Section 14). Seller name, seller id and
    the listing URL are dropped; location is kept at city level. Returns None for sold
    listings and items without a title, id or a price above $0."""
    if first(item, "is_sold") is True:
        return None
    title = first(item, "marketplace_listing_title", "custom_title", "title")
    price = to_price(first(item, "listing_price", "price"))
    source_id = first(item, "id", "listing_id")
    if not (title and source_id and price):
        return None
    desc = first(item, "redacted_description.text", "description.text", "description")
    photos = first(item, "listing_photos", "photos")
    n_photos = (
        len(photos) if isinstance(photos, list) else int(bool(first(item, "primary_listing_photo")))
    )
    return {
        "listing_id": anon_id(source_id),
        "raw_title": str(title).strip(),
        "raw_description": desc.strip() if isinstance(desc, str) else "",
        "price": price,
        "location_city": first(item, "location.reverse_geocode.city", "location.city") or "",
        "delivery_type": delivery_type(first(item, "delivery_types", "delivery_type")),
        "has_clear_photo": n_photos > 0,
    }

## 3. Identify the card, then filter the scope

In [ ]:
#| export
class ExtractedCard(BaseModel):
    listing_id: str
    card_name: str | None = None
    set_name: str | None = None
    number: str | None = None
    variant: Variant | None = None
    condition: Condition = "unknown"
    graded: bool = False
    grade: str | None = None
    language: str = "en"
    is_lot: bool = False
    shipping: float | None = Field(default=None, ge=0.0)
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)

In [ ]:
#| export
class ExtractionBatch(BaseModel):
    cards: list[ExtractedCard]

In [ ]:
#| export
EXTRACT_SYSTEM = """You identify the exact Pokémon trading card in each marketplace listing.
For every listing return:
- card_name: English card name with suffix (e.g. "Charizard", "Charizard ex", "Pikachu VMAX").
- set_name: the set, from the text or implied by the number (4/102 -> "Base Set",
  199/165 -> "151", 44/185 -> "Vivid Voltage"). Null if you can't tell.
- number: the number before the slash, without leading zeros ("044/185" -> "44").
- variant: normal, holo, reverse or 1st edition, if stated.
- condition: NM, LP, MP, HP, DMG, or unknown. "Mint"/"pack fresh" -> NM, "light wear" -> LP,
  "well loved"/"heavily played" -> HP, creases, bends or water damage -> DMG.
- graded/grade: true and the grade if the card is in a PSA, BGS or CGC slab.
- language: two-letter code; "en" unless another language is stated.
- is_lot: true for bundles, lots, binders, boxes, sealed product, or several different cards.
- shipping: shipping cost in USD only if the text states it. Never assume $0.
- confidence: 0-1 that card_name, set_name and number are all correct.
Return one entry per listing, with the same listing_id."""

In [ ]:
#| export
def extract_cards(llm: LLM, rows: list[dict], batch_size: int, log=print) -> dict[str, dict]:
    """Identify the card in each row, `batch_size` listings per LLM call."""
    out: dict[str, dict] = {}
    for i in range(0, len(rows), batch_size):
        batch = [
            {
                "listing_id": r["listing_id"],
                "title": r["raw_title"],
                "description": r["raw_description"],
            }
            for r in rows[i : i + batch_size]
        ]
        prompt = (
            "Listings (JSON):\n<listings>\n"
            + json.dumps(batch, ensure_ascii=False)
            + "\n</listings>"
        )
        res = llm.json(tag="extract", system=EXTRACT_SYSTEM, prompt=prompt, schema=ExtractionBatch)
        out.update({c.listing_id: c.model_dump() for c in res.cards})
        if log:
            log(f"  identified cards in {min(i + batch_size, len(rows))}/{len(rows)} listings")
    return out

In [ ]:
#| export
def build_listings(rows: list[dict], extracted: dict[str, dict]) -> list[CardListing]:
    """Merge anonymized rows with the extracted fields. The row's own fields (id, price)
    always win, so the model can never change a price."""
    listings = []
    for r in rows:
        card = {k: v for k, v in extracted.get(r["listing_id"], {}).items() if v is not None}
        card.pop("listing_id", None)
        listings.append(CardListing.model_validate({**card, **r}))
    return listings

In [ ]:
#| export
def scope_reason(card: CardListing) -> str | None:
    """Why a listing is out of scope (Section 4, step 3), or None if it is in scope."""
    if card.graded:
        return "graded"
    if card.is_lot:
        return "is_lot"
    if card.language.lower() != "en":
        return "non_english"
    if card.card_name == "unknown":
        return "unidentified"
    return None

In [ ]:
#| export
def apply_scope_filter(listings: list[CardListing]) -> tuple[list[CardListing], dict[str, int]]:
    """Drop graded cards, lots/binders, non-English printings and unidentified cards."""
    kept: list[CardListing] = []
    dropped = {"graded": 0, "is_lot": 0, "non_english": 0, "unidentified": 0}
    for card in listings:
        reason = scope_reason(card)
        if reason:
            dropped[reason] += 1
        else:
            kept.append(card)
    return kept, dropped

In [ ]:
#| export
def write_review_sheet(listings: list[CardListing], config: dict[str, Any]) -> Path:
    """CSV for the hand check: fill in checked_ok, fix wrong rows in data/listings.json."""
    min_conf = config["thresholds"]["extraction_confidence_review"]
    out = repo_path(config["data"]["review_sheet"])
    cols = ["listing_id", "raw_title", "price", "card_name", "set_name", "set_code", "number",
            "variant", "condition", "confidence"]  # fmt: skip
    with open(out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([*cols, "needs_review", "checked_ok"])
        for x in listings:
            w.writerow([getattr(x, c) for c in cols] + [x.confidence < min_conf, ""])
    return out

## 4. Market prices

In [ ]:
#| export
class TCGdex:
    """Free TCGdex API (no key). Prices come from TCGplayer through TCGdex."""

    def __init__(self, base_url: str, timeout_s: float = 15):
        self.base = base_url.rstrip("/")
        self.http = httpx.Client(timeout=timeout_s, follow_redirects=True)

    def _get(self, url: str):
        r = self.http.get(url)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()

    def sets(self) -> list[dict]:
        return self._get(f"{self.base}/sets") or []

    def card(self, set_id: str, number: str) -> dict | None:
        for local in dict.fromkeys([number, number.lstrip("0") or "0", number.zfill(3)]):
            data = self._get(f"{self.base}/cards/{set_id}-{local}")
            if data:
                return data
        return None

    @staticmethod
    def price_entry(card: dict) -> dict:
        """Flatten a TCGdex card into the prices.json entry format."""
        tcg = (card.get("pricing") or {}).get("tcgplayer") or {}
        variants = {
            k: (v.get("marketPrice") or v.get("midPrice"))
            for k, v in tcg.items()
            if isinstance(v, dict)
        }
        return {
            "name": card.get("name"),
            "set_name": (card.get("set") or {}).get("name"),
            "variants": {k: v for k, v in variants.items() if v},
            "currency": tcg.get("unit", "USD"),
            "updated": tcg.get("updated"),
            "source": "tcgdex/tcgplayer",
        }

In [ ]:
#| export
SERIES_PREFIX = re.compile(
    r"^(scarlet\s*&\s*violet|sword\s*&\s*shield|sun\s*&\s*moon|xy|black\s*&\s*white|sv|swsh)\W+",
    re.IGNORECASE,
)

In [ ]:
#| export
def resolve_set(name: str, sets: list[dict]) -> str | None:
    """Map a set name to a TCGdex set id: exact match first, then a close fuzzy match.

    Exact-first matters: 'Base Set' must not resolve to 'Base Set 2'. Series prefixes are
    stripped, so 'Scarlet & Violet—151' finds '151'. A name that already is an id passes.
    """
    ids = {s["id"].casefold(): s["id"] for s in sets if s.get("id")}
    if name.casefold() in ids:
        return ids[name.casefold()]
    by_name = {s["name"].casefold(): s["id"] for s in sets if s.get("name")}
    tries = [name, re.split(r"\s*[—–]\s*|\s+-\s+", name)[-1], SERIES_PREFIX.sub("", name)]
    for t in dict.fromkeys(x.strip() for x in tries if x and x.strip()):
        if t.casefold() in by_name:
            return by_name[t.casefold()]
    hit = process.extractOne(name.casefold(), list(by_name), scorer=fuzz.ratio, score_cutoff=88)
    return by_name[hit[0]] if hit else None

In [ ]:
#| export
class PriceBook:
    """Market prices for listings: resolves set ids and fetches missing cards from TCGdex.
    Prices are cached in data/prices.json and the set list in data/tcgdex_sets.json, so each
    card is fetched only once."""

    def __init__(
        self, config: dict[str, Any], prices: dict[str, dict], client: TCGdex | None = None
    ):
        self.prices = prices
        self.client = client or TCGdex(config["tcgdex"]["base_url"], config["tcgdex"]["timeout_s"])
        self.prices_path = repo_path(config["data"]["prices"])
        self.sets_path = repo_path(config["data"]["tcgdex_sets"])
        self._sets: list[dict] | None = None

    def sets(self) -> list[dict]:
        if self._sets is None:
            if self.sets_path.exists():
                self._sets = json.loads(self.sets_path.read_text(encoding="utf-8"))
            else:
                try:
                    self._sets = self.client.sets()
                except httpx.HTTPError as e:
                    raise SourceError(f"TCGdex set list unavailable: {e}") from e
                self.sets_path.parent.mkdir(parents=True, exist_ok=True)
                self.sets_path.write_text(json.dumps(self._sets, indent=1), encoding="utf-8")
        return self._sets

    def resolve_sets(self, listings: list[CardListing]) -> list[str]:
        """Fill set_code from set_name. Returns the set names that could not be matched."""
        unresolved = set()
        for x in listings:
            if x.set_code == "unknown" and x.set_name:
                x.set_code = resolve_set(x.set_name, self.sets()) or "unknown"
                if x.set_code == "unknown":
                    unresolved.add(x.set_name)
        return sorted(unresolved)

    def ensure_prices(self, listings: list[CardListing], pause_s: float = 0.2) -> list[str]:
        """Fetch prices for cards not cached yet. Returns card keys with no TCGplayer price."""
        missing = []
        for key in sorted({x.card_key for x in listings if x.card_key}):
            if key in self.prices:
                continue
            set_id, _, number = key.rpartition("-")
            try:
                card = self.client.card(set_id, number)
            except httpx.HTTPError:
                card = None
            entry = TCGdex.price_entry(card) if card else None
            if entry and entry["variants"]:
                self.prices[key] = entry
            else:
                missing.append(key)
            time.sleep(pause_s)  # be polite to a free API
        return missing

    def save(self) -> None:
        out = {k: v for k, v in self.prices.items() if k != "_meta"}
        out["_meta"] = {
            "source": "TCGdex (TCGplayer market prices)",
            "fetched": time.strftime("%Y-%m-%d"),
        }
        self.prices_path.parent.mkdir(parents=True, exist_ok=True)
        self.prices_path.write_text(json.dumps(out, indent=1, ensure_ascii=False), encoding="utf-8")

## Run it all

In [ ]:
#| export
def save_listings(listings: list[CardListing], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = [x.model_dump() for x in listings]
    path.write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding="utf-8")

In [ ]:
#| export
def run_ingest(
    config: dict[str, Any],
    llm: LLM,
    raw_items: list[dict],
    pricebook: PriceBook | None = None,
    out_path: Path | None = None,
    log=print,
) -> dict[str, Any]:
    """Raw Apify items -> data/listings.json (+ prices, + review sheet). Returns a summary."""
    rows = [r for r in (anonymize(i) for i in raw_items) if r]
    log(f"{len(raw_items)} raw items -> {len(rows)} usable listings")
    extracted = extract_cards(llm, rows, config["ingest"]["batch_size"], log=log)
    kept, dropped = apply_scope_filter(build_listings(rows, extracted))
    log(f"in scope: {len(kept)}, dropped: {dropped}")
    summary: dict[str, Any] = {
        "raw": len(raw_items),
        "usable": len(rows),
        "kept": len(kept),
        "dropped": dropped,
    }
    if pricebook is not None:
        summary["unresolved_sets"] = pricebook.resolve_sets(kept)
        summary["unpriced"] = pricebook.ensure_prices(kept)
        pricebook.save()
        log(f"prices: {len(summary['unpriced'])} cards without a TCGplayer price")
    save_listings(kept, out_path or repo_path(config["data"]["listings"]))
    write_review_sheet(kept, config)
    min_conf = config["thresholds"]["extraction_confidence_review"]
    summary["needs_review"] = [x.listing_id for x in kept if x.confidence < min_conf]
    log(f"rows needing a hand check (confidence < {min_conf}): {len(summary['needs_review'])}")
    return summary

## Load the listing pool

What the Merchant Agent searches: the samples until `data.use_samples` is false.

In [ ]:
#| export
@dataclass
class MarketData:
    """Everything the Merchant Agent searches and checks against."""

    listings: list[CardListing]
    prices: dict[str, dict]
    red_flags: list[RedFlag]
    source: str  # "samples", "snapshot" or "live"
    meta: dict = field(default_factory=dict)

In [ ]:
#| export
def _read_listings(p: Path) -> list[CardListing]:
    return [CardListing.model_validate(x) for x in json.loads(p.read_text(encoding="utf-8"))]

In [ ]:
#| export
def load_data(config: dict[str, Any]) -> MarketData:
    """The sample pool, or the real snapshot once data/listings.json exists and
    `data.use_samples` is false. Live search results are added to the snapshot, never to
    the samples."""
    d = config["data"]
    red_flags = parse_red_flags(repo_path(d["red_flags"]).read_text(encoding="utf-8"))
    use_samples = d["use_samples"] or not repo_path(d["listings"]).exists()
    lp = repo_path(d["sample_listings"] if use_samples else d["listings"])
    pp = repo_path(d["sample_prices"] if use_samples else d["prices"])
    pool = {x.listing_id: x for x in _read_listings(lp)}
    live_path = repo_path(config["merchant"]["live_listings"])
    if not use_samples and live_path.exists():
        pool.update({x.listing_id: x for x in _read_listings(live_path)})
    prices = json.loads(pp.read_text(encoding="utf-8")) if pp.exists() else {}
    meta = prices.pop("_meta", {})
    return MarketData(
        list(pool.values()), prices, red_flags, "samples" if use_samples else "snapshot", meta
    )

## Offline mode

Rule-based stand-ins used when `LIVE = False` (and by the tests).

In [ ]:
#| export
CARD_NAMES = [
    "Charizard ex", "Charizard VMAX", "Charizard V", "Charizard", "Blastoise", "Venusaur",
    "Pikachu VMAX", "Pikachu V", "Pikachu", "Mewtwo", "Zapdos", "Umbreon VMAX", "Umbreon",
    "Gengar", "Mew", "Rayquaza", "Lugia",
]  # fmt: skip
SET_HINTS = [
    (r"\bbase(?: set)?\b", "Base Set"),
    (r"\b151\b", "151"),
    (r"vivid voltage", "Vivid Voltage"),
    (r"evolving skies", "Evolving Skies"),
]
DENOMINATOR_SET = {"102": "Base Set", "165": "151", "185": "Vivid Voltage", "203": "Evolving Skies"}
CONDITION_PATTERNS = [
    (r"\b(nm|near mint|mint)\b", "NM"),
    (r"\b(lp|lightly played|light play)\b", "LP"),
    (r"\b(mp|moderately played)\b", "MP"),
    (r"\b(hp|heavily played|well loved|played)\b", "HP"),
    (r"\b(dmg|damaged|crease|creased|bent|water damage)\b", "DMG"),
]

In [ ]:
#| export
def between(prompt: str, tag: str) -> str:
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", prompt, flags=re.DOTALL)
    return m.group(1) if m else prompt

In [ ]:
#| export
def guess_card_name(text: str) -> str | None:
    low = text.lower()
    return next((n for n in CARD_NAMES if re.search(rf"\b{re.escape(n.lower())}\b", low)), None)

In [ ]:
#| export
def guess_condition(text: str) -> str | None:
    low = text.lower()
    return next((c for p, c in CONDITION_PATTERNS if re.search(p, low)), None)

In [ ]:
#| export
@offline_handler("extract")
def offline_extract(_system: str, prompt: str) -> dict:
    """Rule-based stand-in for the extraction call (offline mode)."""
    cards = []
    for it in json.loads(between(prompt, "listings")):
        text = f"{it.get('title', '')} {it.get('description') or ''}"
        low = text.lower()
        num = re.search(r"\b(\d{1,3})\s*/\s*(\d{2,3})\b", text)
        graded = re.search(r"\b(psa|bgs|cgc)\s*(\d+(?:\.\d)?)", low)
        variant = next((v for k, v in (("reverse", "reverse"), ("1st edition", "1st edition"),
                                       ("holo", "holo")) if k in low), None)  # fmt: skip
        set_name = DENOMINATOR_SET.get(num.group(2)) if num else None
        cards.append({
            "listing_id": it["listing_id"],
            "card_name": guess_card_name(text),
            "set_name": set_name or next((n for p, n in SET_HINTS if re.search(p, low)), None),
            "number": num.group(1).lstrip("0") if num else None,
            "variant": variant,
            "condition": guess_condition(text) or "unknown",
            "graded": bool(graded),
            "grade": graded.group(2) if graded else None,
            "language": "ja" if re.search(r"\b(japanese|jp|jpn)\b", low) else "en",
            "is_lot": bool(re.search(r"\b(lot|bundle|bulk|\d+\s*cards)\b|\+", it.get("title", "").lower())),
            "confidence": 0.9 if num else 0.6,
        })  # fmt: skip
    return {"cards": cards}

# Run it

Offline (default) this runs the whole pipeline on the 28 synthetic raw listings in
`data/samples/sample_raw_listings.json`, writing to a temp folder so nothing real is touched.

To build the real snapshot: set `LIVE = True` (Gemini) and `SCRAPE = True` (Apify, costs
credit; capped by `scrape.max_charge_usd`). Start with `TEST_RUN = True`: 5 listings.
Then hand-check `data/listings_review.csv` and set `data.use_samples: false` in config.yaml.

In [ ]:
import copy
import json
import tempfile

import pandas as pd

from src.config import load_config, repo_path
from src.llm import make_llm

SCRAPE = False  # True: run the Apify actor (needs APIFY_API_TOKEN, costs credit)
TEST_RUN = True  # 5 listings instead of scrape.results_limit

cfg = load_config()
llm = make_llm(cfg)
if not SCRAPE:  # dry run: sample input, temp outputs
    tmp = Path(tempfile.mkdtemp())
    cfg = copy.deepcopy(cfg)
    for key in ("listings", "prices", "review_sheet", "tcgdex_sets"):
        cfg["data"][key] = str(tmp / Path(cfg["data"][key]).name)
    print("dry run, writing to", tmp)

In [ ]:
if SCRAPE:
    s = cfg["scrape"]
    limit = 5 if TEST_RUN else s["results_limit"]
    raw = MarketplaceScraper(cfg).search(
        s["queries"],
        limit=limit,
        max_charge_usd=s["max_charge_usd"],
        details=s["include_listing_details"],
    )
else:
    raw = json.loads(repo_path(cfg["data"]["sample_raw"]).read_text(encoding="utf-8"))
print(len(raw), "raw items; first one anonymized:")
anonymize(raw[0])

Prices: real TCGdex calls when scraping; in the dry run, the sample price file stands in.

In [ ]:
if SCRAPE:
    book = PriceBook(cfg, {})
else:
    sample_prices = json.loads(repo_path(cfg["data"]["sample_prices"]).read_text(encoding="utf-8"))
    sample_prices.pop("_meta", None)

    class SampleTCGdex(TCGdex):  # answers from the sample prices, no network
        def __init__(self):
            pass

        def sets(self):
            return [
                {"id": "base1", "name": "Base Set"},
                {"id": "sv03.5", "name": "151"},
                {"id": "swsh4", "name": "Vivid Voltage"},
            ]

        def card(self, set_id, number):
            e = sample_prices.get(f"{set_id}-{number}")
            return e and {
                "name": e["name"],
                "set": {"name": e["set_name"]},
                "pricing": {"tcgplayer": {k: {"marketPrice": v} for k, v in e["variants"].items()}},
            }

    book = PriceBook(cfg, {}, client=SampleTCGdex())

summary = run_ingest(cfg, llm, raw, pricebook=book)
summary

In [ ]:
listings = json.loads(repo_path(cfg["data"]["listings"]).read_text(encoding="utf-8"))
cols = [
    "listing_id",
    "raw_title",
    "price",
    "card_name",
    "set_name",
    "set_code",
    "number",
    "condition",
    "confidence",
]
pd.DataFrame(listings)[cols]

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()